# Thêm Thư Viện

In [1]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [3]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2025;'
)
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=Library_DWH;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2025;'
)

## Đọc data từ SQL Server

In [4]:
query_NienKhoa = "SELECT DISTINCT dbo.DecodeUTF8String(Khoa_hoc) AS Khoa_hoc FROM Ban_doc" # Đọc dữ liệu từ bảng Ban_doc trong CSDL libol
df_nienkhoa = pd.read_sql(query_NienKhoa, conn_libol)
print(df_nienkhoa)

C:\Users\admin\AppData\Local\Temp\ipykernel_13804\1089893420.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_nienkhoa = pd.read_sql(query_NienKhoa, conn_libol)


         Khoa_hoc
0     2000 - 2005
1            2018
2    2012  - 2014
3             000
4     2023 - 2026
..            ...
243     2018-2022
244            98
245     2003 - 08
246   2012 - 2016
247     2015-2017

[248 rows x 1 columns]


## Xử lý data

In [5]:
df_nienkhoa['Khoa_hoc'] = df_nienkhoa['Khoa_hoc'].str.replace(r'\s*-\s*', '-', regex=True)
df_nienkhoa['Khoa_hoc'] = df_nienkhoa['Khoa_hoc'].replace('', np.nan)
df_nienkhoa = df_nienkhoa.dropna(subset=['Khoa_hoc'])
print(df_nienkhoa)

      Khoa_hoc
0    2000-2005
1         2018
2    2012-2014
3          000
4    2023-2026
..         ...
243  2018-2022
244         98
245    2003-08
246  2012-2016
247  2015-2017

[246 rows x 1 columns]


In [6]:
df_nienkhoa['ID'] = range(1, len(df_nienkhoa) + 1)
new_row = pd.DataFrame({'ID': [0], # Tạo hàng dữ liệu giả lập cho thư viện không xác định
                        'Khoa_hoc': ['(Không xác định)']})
df_nienkhoa = pd.concat([df_nienkhoa, new_row], ignore_index=True) # Thêm vào dataframe
df_nienkhoa = df_nienkhoa.sort_values(by="ID", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn
print(df_nienkhoa)

             Khoa_hoc   ID
0    (Không xác định)    0
1           2000-2005    1
2                2018    2
3           2012-2014    3
4                 000    4
..                ...  ...
242         2018-2022  242
243                98  243
244           2003-08  244
245         2012-2016  245
246         2015-2017  246

[247 rows x 2 columns]


## Load data

### [Nếu cần] Clear bảng

In [7]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM olap.DIM_Nien_khoa"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [8]:
cursor_dwh = conn_dwh_library.cursor()
insert_query = """
                INSERT INTO olap.DIM_Nien_khoa (ID_nien_khoa, Ten_nien_khoa) 
                VALUES (?, ?)
                """
for index, row in df_nienkhoa.iterrows():
    values = (row['ID'], 
              row['Khoa_hoc'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_library.commit()